# Lakebase Search execution evidence

Calls the deployed Nimbus app with a short-lived token obtained from a named Databricks CLI profile. The token is held only in memory and is never printed or stored in notebook output.

In [ ]:
import json, os, subprocess, urllib.parse, urllib.request

PROFILE = os.environ.get("DATABRICKS_CONFIG_PROFILE", "fe-sandbox-last-penguin")
APP_NAME = "nimbus-growth-desk"

def cli_json(*args):
    raw = subprocess.check_output(["databricks", *args, "--profile", PROFILE, "-o", "json"], text=True)
    return json.loads(raw)

app = cli_json("apps", "get", APP_NAME)
token = cli_json("auth", "token")["access_token"]
url = app["url"].rstrip("/") + "/api/search-experiments?" + urllib.parse.urlencode({"q": "checkout android gen-z", "limit": 5})
request = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}"})
with urllib.request.urlopen(request, timeout=60) as response:
    result = json.load(response)
result

In [ ]:
plan = "\n".join(result["execution_plan"])
ids = [row["experiment_id"] for row in result["rows"]]
assert result["source_table"] == "nimbus_serving.experiments"
assert result["search_function"] == "app.search_experiments"
assert result["index"] == "experiments_description_bm25_idx"
assert result["read_only"] is True
assert "Index Scan" in plan
assert "EXP-0000009" in ids
print(f"executed_at: {result['executed_at']}")
print(f"path: {result['source_table']} -> {result['search_function']} -> {result['index']} -> Index Scan -> EXP-0000009")
print(f"rows: {ids}")
print("SEARCH_EXECUTION_VERIFIED")